# 🛋️ FurnForge AI — Google Colab GPU Backend

This notebook runs the **FastAPI GPU backend** for FurnForge AI (Stable Diffusion 1.5 + ControlNet Canny + LCM-LoRA + TripoSR 3D).

### ⚠️ Important Before Starting:
Make sure your Colab is using a GPU: **Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save**

In [ ]:
# ========================================================================
# STEP 1: Mount Drive, Clone Repository & Install Dependencies
# ========================================================================
from google.colab import drive
import os, shutil, sys

%cd /content

# Mount Google Drive to persist generated images and database across sessions
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# Clone fresh repository
REPO_DIR = '/content/FurnForgeAI'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR, ignore_errors=True)
!git clone https://github.com/yandrapragada-hari/FurnForgeAI.git {REPO_DIR}

%cd {REPO_DIR}/backend

# Link persistent outputs, uploads, and database to Google Drive
PERSIST_DIR = '/content/drive/MyDrive/FurnForgeAI'
for rel_path in ['outputs', 'uploads']:
    folder = os.path.join(REPO_DIR, 'backend', rel_path)
    target = os.path.join(PERSIST_DIR, rel_path)
    os.makedirs(target, exist_ok=True)
    if os.path.islink(folder):
        os.unlink(folder)
    elif os.path.exists(folder):
        shutil.rmtree(folder)
    os.symlink(target, folder)

db_file = os.path.join(REPO_DIR, 'backend/app/database/furnforge.db')
drive_db = os.path.join(PERSIST_DIR, 'furnforge.db')
if os.path.exists(db_file) and not os.path.exists(drive_db):
    shutil.copy2(db_file, drive_db)
if os.path.exists(db_file):
    os.remove(db_file)
if os.path.exists(drive_db):
    os.symlink(drive_db, db_file)

# Install Linux dependencies and Cloudflare Tunnel
!apt-get update -qq && apt-get install -y -qq libgl1 libglib2.0-0 psmisc wget > /dev/null 2>&1
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Install PyTorch with CUDA 12.1 + FurnForge AI Python dependencies
!pip install -q --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -q peft opencv-python-headless

# Verify GPU
import torch
print('\n' + '=' * 55)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'🎉 GPU ACTIVE: {gpu_name} ({vram:.1f} GB VRAM)')
    print('✅ All packages installed and ready!')
else:
    print('⚠️ STILL RUNNING ON CPU!')
    print('👉 Go to: Runtime -> Change runtime type -> T4 GPU -> Save, then re-run.')
print('=' * 55)

In [ ]:
# ========================================================================
# STEP 2: Pre-cache 3D TripoSR Neural Weights (1.7 GB)
# (Run this once so 3D generation is fast and never times out in the UI)
# ========================================================================
from huggingface_hub import hf_hub_download
import rembg

print('⬇️ Downloading and verifying TripoSR 3D weights (1.7 GB)...')
hf_hub_download('stabilityai/TripoSR', 'model.ckpt')
hf_hub_download('stabilityai/TripoSR', 'config.yaml')
rembg.new_session('u2net')
print('✅ TripoSR 3D weights cached! 3D reconstruction will take ~10-15s.')

In [ ]:
# ========================================================================
# STEP 3: Launch FurnForge AI Backend & Cloudflare Tunnel
# (Keep this cell running while using the web app)
# ========================================================================
import os, sys, time, subprocess, re, urllib.request, threading

%cd /content/FurnForgeAI/backend
!git pull origin main

# 1. Clean up any stale instances
!fuser -k 8005/tcp 2>/dev/null || true
!killall cloudflared 2>/dev/null || true
!pkill -9 -f uvicorn 2>/dev/null || true
time.sleep(1)

# 2. Launch FastAPI backend
print('⚡ Launching FurnForge backend on port 8005...')
backend_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8005'],
    cwd='/content/FurnForgeAI/backend',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

def stream_logs(proc):
    for line in proc.stdout:
        print(f'[Backend] {line.strip()}')

threading.Thread(target=stream_logs, args=(backend_proc,), daemon=True).start()

# 3. Verify health
is_ready = False
for _ in range(25):
    time.sleep(1)
    try:
        with urllib.request.urlopen('http://localhost:8005/api/health', timeout=2) as r:
            if r.status == 200:
                is_ready = True
                break
    except Exception:
        pass

if not is_ready:
    print('❌ Backend failed to start. Check backend logs above.')
else:
    print('✅ Backend is healthy and responding on port 8005!')
    print('🌐 Connecting Cloudflare Tunnel...')
    tunnel_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:8005'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    tunnel_url = None
    for line in tunnel_proc.stdout:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break

    if tunnel_url:
        print('\n' + '=' * 65)
        print('🎉 SUCCESS! YOUR COLAB BACKEND IS LIVE AT:')
        print(f'👉  {tunnel_url}')
        print('=' * 65)
        print(f'\n📋 Copy this URL into Vercel -> Environment Variables -> VITE_BACKEND_URL:')
        print(f'   {tunnel_url}\n')
        print('⏳ Keeping backend alive (leave this cell running)...\n')

    # 4. Auto-restart watcher to ensure 100% uptime
    try:
        while True:
            time.sleep(5)
            if backend_proc.poll() is not None:
                print('⚠️ Backend exited! Auto-restarting...')
                backend_proc = subprocess.Popen(
                    [sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8005'],
                    cwd='/content/FurnForgeAI/backend',
                    stdout=subprocess.PIPE,
                    stderr=subprocess.STDOUT,
                    text=True,
                    bufsize=1
                )
                threading.Thread(target=stream_logs, args=(backend_proc,), daemon=True).start()
    except KeyboardInterrupt:
        print('\n🛑 Stopping server and tunnel...')
        backend_proc.terminate()
        tunnel_proc.terminate()